# Practice 6. Tuning a PID controller with a genetic algorithm and a particle swarm

Industrial AI, KBTU. Goes with Lecture 10 and `inclass_work_10.ipynb`. **1.25 points.**

### The plant and what you have

A shell-and-tube heat exchanger heats process water with steam. A PID controller holds the outlet
temperature by moving the steam valve. The operating point: valve at 40 %, outlet at 65 °C. The
controller has to bring the water 5 °C hotter after a setpoint change, and stay calm when the steam
supply or the flow changes.

- `exchanger_step_test.csv`, in `data/exchanger/` of the course repository: an open-loop step test
  made by the plant engineers. With the controller in manual, the valve was moved from 40 to 50 %
  at $t$ = 60 s and the outlet temperature was logged every second for 15 minutes. Columns
  `time_s`, `valve_pct`, `outlet_temp_C`.
- `exchanger.py`, next to this notebook: the exchanger itself, in a form you can run.
  `closed_loop(Kp, Ki, Kd)` puts your PID on it and returns the response to the +5 °C step. In
  this practice it plays the real plant. **You tune on your model and only check on the
  exchanger**, as on a real plant, where every closed-loop test costs production time. Do not
  read its parameters.

### How the tasks are built

Each task says what to write: the functions, their arguments and what they return, and the
variables that hold your results. How you write them is up to you, as long as you use the method
the task names. Under your cell sits a cell of **tests**. Run it after your code: it either prints
`all tests passed` or stops at the first check that failed and says what is wrong. The tests check
what your code does on small cases with known answers, not how it is written.

These tests are public. When you hand in, the same tests run again together with hidden ones, on
other inputs and on your results, and the share of tests passed gives the points for the code.
Code that only fits the public cases loses those points.

Cells marked *Common ground* are written for you. Run them as they are: they fix the model, the
simulator, the cost, the counter of evaluations and the drivers of GA and PSO, so that everybody's
numbers can be compared.

The methods are yours: the two-point identification, the step measures, SIMC, the ultimate gain by
bisection and Ziegler–Nichols, the GA operators, the PSO step and the robust cost.
`scipy.optimize.curve_fit` is allowed in Task 1 (and in Task 8); `scipy.optimize` is not used
anywhere else.

### What you hand in

Two files:

1. this notebook, run top to bottom, named `Surname_Name_practice_6.ipynb`;
2. `practice_6_answers.md` with your answers to the questions under each task, named
   `Surname_Name_practice_6_answers.md`.

Every answer needs the numbers your notebook printed. A number the notebook does not print scores
zero, and so does a number printed with nothing said about it.

| Task | 1 | 2 | 3 | 4 | 5 | 6 | 7 | total |
|---|---|---|---|---|---|---|---|---|
| code (tests) | 0.08 | 0.08 | 0.08 | 0.13 | 0.13 | 0.06 | 0.07 | 0.63 |
| answers | 0.07 | 0.07 | 0.07 | 0.12 | 0.12 | 0.09 | 0.08 | 0.62 |

Task 8 at the end is optional and gives no points: extra work for those who want it. A
physics-informed neural network (PINN) finds a better model of the exchanger from the same step
test. It needs PyTorch (`pip install torch`; the CPU build is enough, and a GPU is used if PyTorch
finds one). Tasks 1–7 do not need it, and skipping Task 8 costs nothing.

### The rules

- **Tune on the model, check on the exchanger.** `closed_loop` is called only in Tasks 6, 7 and 8.
- Every tuning is reported with all three gains and with its overshoot, settling time and ITAE.
- GA and PSO get the same budget, 1500 evaluations of the cost per run, counted in code with
  `CostCounter`.
- Every optimiser runs with a fixed seed, and the seed is printed next to its result.
- Do not rename the functions and variables the tests use, and do not edit the test cells.
- Before handing in, restart the kernel and run all cells; Tasks 1–7 run in about five minutes on a
  laptop, Task 8 adds two or three. Then check your answers against the new output.
- Figures need axis labels and a title that says what to look at.
- Stuck? `inclass_work_10.ipynb` does every step on a furnace.

In [2]:
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

from scipy.optimize import curve_fit

BLUE, ORANGE, RED, GREEN, PURPLE, GREY = (
    "#2a6f8e", "#e07b39", "#c0392b", "#4a8f5a", "#9b3f9b", "#9aa0a6")

mpl.rcParams.update({
    "figure.dpi": 110,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#cccccc",
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e6e6e6",
    "grid.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.labelsize": 10,
    "font.size": 10,
    "font.family": ["Segoe UI", "DejaVu Sans", "sans-serif"],
    "legend.frameon": False,
    "lines.linewidth": 1.6,
})

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

KAGGLE = {
    'exchanger': None,
}


def dataset_folder(name):
    """The folder with the files of a data set: the Kaggle copy (downloaded once, then cached),
    or data/<name>/ of the course repo if Kaggle is out of reach."""
    if KAGGLE[name]:
        try:
            import contextlib, io, logging, kagglehub
            logging.getLogger("kagglehub").setLevel(logging.ERROR)
            with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                folder = Path(kagglehub.dataset_download(KAGGLE[name]))
            print(f"{name}: Kaggle {KAGGLE[name]}")
            return folder
        except Exception as exc:
            print(f"Kaggle unavailable ({type(exc).__name__}), looking for data/{name}/")
    for root in [Path.cwd(), *Path.cwd().parents]:
        if (root / "data" / name).is_dir():
            return root / "data" / name
    raise FileNotFoundError(f"no data/{name}/ here or above; run data/fetch.py in the course repo")

print(" | ".join(f"{m} {sys.modules[m].__version__}" for m in
                 ("numpy", "pandas", "matplotlib", "scipy", "sklearn", "torch") if m in sys.modules))

numpy 2.5.5 | pandas 3.1.1 | matplotlib 3.11.2 | scipy 1.18.1 | torch 2.8.0+cu128


In [3]:
step_test = pd.read_csv(dataset_folder("exchanger") / "exchanger_step_test.csv")
T_STEP, DU = 60.0, 10.0          # the test: the valve moved by +10 % at t = 60 s
print(step_test.shape)
step_test.head()

(900, 3)


,time_s,valve_pct,outlet_temp_C
0,0.000,40.000,65.024
1,1.000,40.000,64.917
2,2.000,40.000,65.060
3,3.000,40.000,65.075
4,4.000,40.000,64.844


---

## Task 1. A model from the step test  
*0.15 points: 0.08 for the code (tests), 0.07 for the answers*

A **first order plus dead time** (FOPDT) model: after a valve step $\Delta u$ at $t_s$ the output
starts at $y_0$, waits $L$ seconds, then moves towards $y_0 + K\,\Delta u$ with the time constant $T$:

$$y(t) = y_0 + K\,\Delta u\,\bigl(1 - e^{-(t - t_s - L)/T}\bigr) \quad \text{for } t > t_s + L, \qquad y(t) = y_0 \text{ before}.$$

The **two-point method** (Smith) reads the three numbers off a step response:

| step | rule |
|---|---|
| gain | $K = (y_\infty - y_0) / \Delta u$, with $y_\infty$ the mean of the last 60 samples and $y_0$ the mean before the step |
| two times | $t_{28}$ and $t_{63}$: the first times after the step when the output has made 28.3 % and 63.2 % of its total change, counted from $t_s$ |
| lag and delay | $T = 1.5\,(t_{63} - t_{28})$, $L = t_{63} - T$ |

For a true FOPDT the response after the dead time is $1 - e^{-(t - L)/T}$, which reaches 28.3 % at
$L + T/3$ and 63.2 % at $L + T$. The gap between the two points is $\frac{2}{3}T$, and the formulas
follow. Two points far apart on the curve are less sensitive to noise than one tangent at the
steepest point.

**Write**

- `two_point(t, y, t_step, du)`: `(K, T, L)` as floats, for arrays `t` and `y` of one step
  response, the step `du` at `t_step`; the step may be negative;
- `FOPDT_2P`: `two_point` on the step test, with the temperature smoothed by a centred moving average
  of 9 samples (`rolling(9, center=True, min_periods=1)`);
- `FOPDT_LS`: `(K, T, L)` from `curve_fit` of the FOPDT response above to the *raw* temperature,
  started from `FOPDT_2P`, with $y_0$ the mean before the step.

From Task 2 on, the common cell fixes the model at $K$ = 0.80 °C/%, $T$ = 101 s, $L$ = 36 s.

In [ ]:
def two_point(t, y, t_step, du):
    ...


FOPDT_2P = ...
FOPDT_LS = ...

In [ ]:
# Tests for Task 1 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_exact_fopdt_response():
    t_chk = np.arange(0, 600, 0.1)
    s_chk = np.clip(t_chk - 20 - 12, 0, None)         # time after the dead time
    y_chk = 10 + 2 * 4 * (1 - np.exp(-s_chk / 50))           # K 2, T 50 s, L 12 s
    got = two_point(t_chk, y_chk, 20.0, 4.0)
    ok = np.allclose(got, [2, 50, 12], atol=0.15)
    assert ok, f"K = 2, T = 50, L = 12 expected, got {np.round(got, 2)}"


test_exact_fopdt_response()


def test_negative_step():
    t_chk = np.arange(0, 300, 0.1)
    s_chk = np.clip(t_chk - 10 - 5, 0, None)
    y_chk = 3 - 1.5 * 2 * (1 - np.exp(-s_chk / 20))          # K -1.5, T 20 s, L 5 s
    got = two_point(t_chk, y_chk, 10.0, 2.0)
    ok = np.allclose(got, [-1.5, 20, 5], atol=0.15)
    assert ok, f"K = -1.5, T = 20, L = 5 expected, got {np.round(got, 2)}"


test_negative_step()


def test_models_reported():
    for name, p in {"FOPDT_2P": FOPDT_2P, "FOPDT_LS": FOPDT_LS}.items():
        positive = len(p) == 3 and all(np.isfinite(v) and v > 0 for v in p)
        assert positive, f"{name}: three positive numbers K, T, L"


test_models_reported()


print("all tests passed")

**Questions** (answer them in `practice_6_answers.md`, Task 1, with the numbers your code printed):

1. Plot the step test: valve and outlet temperature against time. Report $K$, $T$, $L$ of the two-point method on the smoothed curve and of least squares, with the RMS residual of each model.
2. Plot the data with both fitted curves, with an inset on the first two minutes. Where does a first order plus dead time model miss this exchanger? Which of the two sets would you use, and why are the two dead times different?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Common ground for Tasks 2–8

Run this cell as it is. It fixes the model of Task 1 for everybody ($K$ = 0.80, $T$ = 101 s,
$L$ = 36 s) and gives `simulate_loop(Kp, Ki, Kd, L=...)`: the PID of In-class work 10 (derivative on
the measurement with a filter, anti-windup) on that model, for a +5 °C setpoint step, 15 minutes at
1 s. Gains may be arrays, one loop per element; it returns `t`, `Y`, `U` with one column per loop.
`K`, `T` and `L` can be changed to run the loop on another FOPDT. The exchanger in `exchanger.py`
runs under the same `_pid_loop`.

In [7]:
from exchanger import closed_loop

K_M, T_M, L_M = 0.80, 101.0, 36.0      # the FOPDT model of Task 1, the same for everybody
R_STEP = 5.0                         # setpoint step, °C
U_LO, U_HI = -40.0, 60.0               # valve travel around 40 %, %

def _pid_loop(Kp, Ki, Kd, plant_step, n_states, d, t_end, dt, r, u_lo, u_hi):
    # PID (derivative on the measurement, filtered; anti-windup) on any plant.
    # plant_step(x, u) advances the state x (n_states, loops) by dt; the output is its
    # last row. d: dead time in steps. Gains may be arrays: one loop per element.
    # Returns t (steps,), Y and U (steps, loops).
    gains = [np.atleast_1d(np.asarray(g, float)) for g in (Kp, Ki, Kd)]
    Kp, Ki, Kd = np.broadcast_arrays(*gains)
    n = Kp.size
    steps = int(t_end / dt)
    Tf = Kd / np.maximum(Kp, 1e-9) / 10           # derivative filter time constant, s

    x = np.zeros((n_states, n))
    integral, deriv, y_prev = np.zeros(n), np.zeros(n), np.zeros(n)
    valve_queue = np.zeros((d + 1, n))            # commands on their way to the plant
    Y, U = np.zeros((steps, n)), np.zeros((steps, n))
    for k in range(steps):
        y = x[-1]
        e = r - y

        # filtered derivative of the measurement, then the command within the valve limits
        if k:
            deriv = (Tf * deriv - Kd * (y - y_prev)) / (Tf + dt)
        y_prev = y
        u_raw = Kp * e + integral + deriv
        u = np.clip(u_raw, u_lo, u_hi)

        # anti-windup: no integration towards a limit the valve already sits at
        winding = (u != u_raw) & (np.sign(e) == np.sign(u_raw))
        integral = integral + np.where(winding, 0.0, Ki * e * dt)

        # the plant sees the command sent d steps ago
        valve_queue = np.roll(valve_queue, -1, axis=0)
        valve_queue[-1] = u
        x = plant_step(x, valve_queue[0])
        Y[k], U[k] = x[-1], u
    return np.arange(1, steps + 1) * dt, Y, U


def simulate_loop(Kp, Ki, Kd, K=K_M, T=T_M, L=L_M, r=R_STEP, t_end=900.0, dt=1.0,
                  u_lo=U_LO, u_hi=U_HI):
    # PID on the FOPDT model: a +5 °C setpoint step. Gains may be arrays: a loop each.
    a = np.exp(-dt / T)
    plant_step = lambda x, u: a * x + (1 - a) * K * u
    d = int(round(L / dt))                                 # dead time in steps
    return _pid_loop(Kp, Ki, Kd, plant_step, 1, d, t_end, dt, r, u_lo, u_hi)


t_demo, Y_demo, U_demo = simulate_loop([1.0, 2.0], [0.01, 0.02], [0.0, 30.0])
_, Y_plant_demo, _ = closed_loop(1.0, 0.01, 0.0)
print("simulate_loop works:", Y_demo.shape, "| closed_loop works:", Y_plant_demo.shape)

simulate_loop works: (900, 2) | closed_loop works: (900, 1)


---

## Task 2. Scoring a response, and the SIMC baseline  
*0.15 points: 0.08 for the code (tests), 0.07 for the answers*

For one response $y(t)$ to a setpoint step of size $r$, sampled every $\Delta t$:

| measure | formula |
|---|---|
| overshoot, % | $\max(0,\ \max_k y_k - r) / r \cdot 100$ |
| settling time, s | the last $t_k$ with $\lvert r - y_k \rvert > 0.02\,r$; `np.inf` if the response is still outside the band at the last sample, 0 if it never left it |
| ITAE | $\sum_k t_k\,\lvert r - y_k \rvert\,\Delta t$ |

The **SIMC** rule (Skogestad, 2003) gives a PI controller for an FOPDT model, with one tuning
knob, the desired closed-loop time constant $\tau_c$:

$$K_p = \frac{T}{K(\tau_c + L)}, \qquad T_i = \min\bigl(T,\ 4(\tau_c + L)\bigr), \qquad K_i = \frac{K_p}{T_i}, \qquad K_d = 0.$$

$\tau_c = L$ is the usual choice: as fast as the dead time allows, without much overshoot.

**Write**

- `step_metrics(t, y, r=R_STEP)`: a dict with the keys `"overshoot"`, `"settling"`, `"ITAE"`, each a
  Python `float`, for arrays `t` and `y` of one response;
- `simc(K, T, L, tau_c)`: the tuple `(Kp, Ki, 0.0)`;
- `SIMC = simc(K_M, T_M, L_M, L_M)`; Tasks 3, 6 and 7 need it.

In [ ]:
def step_metrics(t, y, r=R_STEP):
    ...


def simc(K, T, L, tau_c):
    ...


SIMC = ...

In [ ]:
# Tests for Task 2 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_metrics_of_a_made_up_response():
    t_chk = np.arange(1, 401, 1.0)
    y_chk = 5 * (1 - np.exp(-t_chk / 40) * np.cos(t_chk / 25))
    got = step_metrics(t_chk, y_chk)
    has_keys = isinstance(got, dict) and {"overshoot", "settling", "ITAE"} <= set(got)
    assert has_keys, "a dict with overshoot, settling, ITAE"
    overshoot, settling, itae = got["overshoot"], got["settling"], got["ITAE"]
    msg = f"overshoot: expected 16.87 %, got {overshoot:.2f}"
    assert abs(overshoot - 16.87) < 0.01, msg
    assert settling == 156.0, f"settling: expected 156 s, got {settling}"
    assert abs(itae - 4854.97) < 0.1, f"ITAE: expected 4854.97, got {itae:.2f}"


test_metrics_of_a_made_up_response()


def test_settling_at_the_edges():
    t_chk = np.arange(1, 101, 1.0)
    inside = step_metrics(t_chk, np.full(100, 5.05))
    ok = inside["settling"] == 0 and np.isclose(inside["overshoot"], 1.0)
    assert ok, "never outside the 2 % band: settling 0; overshoot 1 %"
    outside = step_metrics(t_chk, np.full(100, 4.5))
    ok = outside["settling"] == np.inf and outside["overshoot"] == 0
    assert ok, "outside the band at the end: settling inf; below r: overshoot 0"


test_settling_at_the_edges()


def test_simc_both_branches():
    long_lag = simc(2.0, 50.0, 5.0, 5.0)
    short_lag = simc(1.0, 20.0, 10.0, 10.0)
    ok = np.allclose(long_lag, (2.5, 0.0625, 0.0))
    assert ok, "T > 4 (tau_c + L): Ti = 40 s, Kp = 2.5"
    ok = np.allclose(short_lag, (1.0, 0.05, 0.0))
    assert ok, "T < 4 (tau_c + L): Ti = T = 20 s, Kp = 1"
    assert len(SIMC) == 3 and SIMC[2] == 0, "SIMC: a PI tuning (Kp, Ki, 0.0)"


test_simc_both_branches()


print("all tests passed")

**Questions** (answer them in `practice_6_answers.md`, Task 2, with the numbers your code printed):

1. Report the SIMC gains and their overshoot, settling time and ITAE on the model.
2. Run the same loop with $K_p$ twice as large. What happens to each of the three measures, and why?
3. Which of the measures would you show to the plant manager, and which to an optimiser? Why?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 3. The Ziegler–Nichols baseline  
*0.15 points: 0.08 for the code (tests), 0.07 for the answers*

Ziegler and Nichols (1942) tune from the edge of stability. Close the loop with a P controller and
raise $K_p$ until the oscillation neither grows nor decays: that gain is the **ultimate gain**
$K_u$, and the period of the oscillation is the **ultimate period** $P_u$. Then

$$K_p = 0.6\,K_u, \qquad T_i = P_u / 2, \qquad T_d = P_u / 8, \qquad K_i = K_p / T_i, \qquad K_d = K_p\,T_d.$$

On a real plant this experiment is risky; on a model it is a bisection. A P-only loop without
valve limits, `simulate_loop(Kp, 0, 0, K=K, T=T, L=L, u_lo=-1e9, u_hi=1e9, t_end=2000)`, **grows** when
the peak-to-peak amplitude of the last 500 samples is larger than that of the 500 samples before.
An unstable loop overflows to `nan`, and every comparison with `nan` is False: write the test as
`not (new <= old)` so that `nan` counts as growing. Bisect between $K_p$ = 0.1 and 20; 40 halvings
are plenty. $P_u$ is the mean distance between the peaks (local maxima) of the response at $K_u$
from sample 1000 on.

**Write**

- `ultimate_gain(K, T, L)`: `(Ku, Pu)` of the FOPDT model with these parameters, by bisection;
- `zn_pid(Ku, Pu)`: the tuple `(Kp, Ki, Kd)` of the rule above;
- `ZN = zn_pid(*ultimate_gain(K_M, T_M, L_M))`.

In [ ]:
def ultimate_gain(K, T, L):
    ...


def zn_pid(Ku, Pu):
    ...


ZN = ...

In [ ]:
# Tests for Task 3 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_zn_rule():
    got = zn_pid(2.0, 100.0)
    ok = np.allclose(got, (1.2, 0.024, 15.0))
    assert ok, f"Ku = 2, Pu = 100 s: expected (1.2, 0.024, 15), got {got}"


test_zn_rule()


def test_ultimate_gain_is_the_edge():
    Ku, Pu = ultimate_gain(1.0, 50.0, 10.0)


    def ptp_ratio(Kp):
        # growth of the oscillation of a P-only loop from the third to the fourth 500 s
        t, Y, _ = simulate_loop(Kp, 0, 0, K=1.0, T=50.0, L=10.0,
                                u_lo=-1e9, u_hi=1e9, t_end=2000)
        y = Y[:, 0]
        return np.ptp(y[1500:]) / np.ptp(y[1000:1500])


    ok = ptp_ratio(0.95 * Ku) < 1 < ptp_ratio(1.05 * Ku)
    assert ok, "below Ku the oscillation decays, above it grows"
    msg = f"for a dead time of 10 s the period lies between 2L and 4L, got {Pu:.1f}"
    assert 20 < Pu < 40, msg


test_ultimate_gain_is_the_edge()


def test_zn_reported():
    assert len(ZN) == 3 and all(v > 0 for v in ZN), "ZN: a PID tuning (Kp, Ki, Kd)"


test_zn_reported()


print("all tests passed")

**Questions** (answer them in `practice_6_answers.md`, Task 3, with the numbers your code printed):

1. Report $K_u$, $P_u$ and the Ziegler–Nichols gains.
2. Print a table of ZN and SIMC with the three gains and the three measures on the model, and plot both responses. Which is the better baseline for the optimisers to beat, and why must the comparison include both?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Common ground for Tasks 4–7

Run this cell as it is. It fixes the search box, the budget and the cost. `itae_cost(P, L=...)`
takes an array of tunings, one row `[Kp, Ki, Kd]` each, and returns their ITAE on the model,
calling your `step_metrics`. `CostCounter` wraps a cost and counts how many tunings it evaluated:
that is the budget GA and PSO must share.

In [14]:
BOUNDS_LO = np.array([0.1, 0.0, 0.0])          # Kp, Ki, Kd
BOUNDS_HI = np.array([20.0, 0.2, 300.0])
POP, GENERATIONS = 30, 50                # 1500 evaluations per run, for GA and for PSO


def itae_cost(P, L=L_M):
    # ITAE of every row of P = [Kp, Ki, Kd] on the model with the dead time L.
    # (k, 3) -> (k,)
    t, Y, _ = simulate_loop(P[:, 0], P[:, 1], P[:, 2], L=L)
    return np.array([step_metrics(t, Y[:, j])["ITAE"] for j in range(Y.shape[1])])


class CostCounter:
    # Wraps a cost of an array of tunings, (k, 3) -> (k,), and counts the tunings
    # it evaluated.
    def __init__(self, cost):
        self.cost, self.evaluations = cost, 0

    def __call__(self, P):
        self.evaluations += len(P)
        return self.cost(P)


box = pd.DataFrame({"low": BOUNDS_LO, "high": BOUNDS_HI}, index=["Kp", "Ki", "Kd"]).T
print(box.to_string())
print(f"budget: {POP} x {GENERATIONS} = {POP * GENERATIONS} evaluations per run")

         Kp    Ki      Kd
low   0.100 0.000   0.000
high 20.000 0.200 300.000
budget: 30 x 50 = 1500 evaluations per run


*Common ground for Task 4: the genetic algorithm of In-class work 10, Part 5, except for its two
operators, which you write. Tournament selection, elitism, `pop * generations` evaluations. Run it
as it is.*

In [15]:
def run_ga(cost, lo, hi, pop=POP, generations=GENERATIONS, alpha=0.25, p_mut=0.2, elite=2,
           seed=0):
    # Returns the best tuning, its cost and the best cost after each generation.
    rng = np.random.default_rng(seed)
    P = lo + (hi - lo) * rng.random((pop, len(lo)))
    f = cost(P)
    curve = [f.min()]

    def parent():
        # tournament of two: the better of two random individuals
        i, j = rng.integers(0, pop, 2)
        return P[i] if f[i] < f[j] else P[j]

    for _ in range(generations - 1):
        children = [P[i] for i in np.argsort(f)[:elite]]          # elitism
        while len(children) < pop:
            a = parent()
            b = parent()
            child = blend_crossover(a, b, alpha, rng)
            children.append(mutate(child, lo, hi, p_mut, rng))
        P = np.array(children)
        f = cost(P)
        curve.append(min(curve[-1], f.min()))
    return P[np.argmin(f)], float(f.min()), np.array(curve)

---

## Task 4. A genetic algorithm  
*0.25 points: 0.13 for the code (tests), 0.12 for the answers*

The two operators of the GA:

| operator | rule |
|---|---|
| blend crossover (BLX-$\alpha$) | the child is $\mathbf{a} + \mathbf{w} \odot (\mathbf{b} - \mathbf{a})$, every component of $\mathbf{w}$ drawn uniformly from $[-\alpha,\ 1 + \alpha]$ |
| mutation | every gene, with probability `p_mut`, is shifted by a normal random number with standard deviation 10 % of its range `hi - lo`; the child is then clipped to the box |

With $\alpha = 0$ the child lies in the box spanned by its parents; with $\alpha > 0$ it can step a
little beyond them, so the population does not shrink just from averaging. Mutation keeps the
search from settling too early: without it the GA can only recombine what the first population
had.

**Write**

- `blend_crossover(a, b, alpha, rng)`: the child, an array of the same shape as the parents `a`, `b`;
- `mutate(child, lo, hi, p_mut, rng)`: the mutated child, clipped to `[lo, hi]`;

both draw every random number from the generator `rng` they are given, and leave their arguments
unchanged;

- `ga_table`: a `pd.DataFrame` indexed by seed 0–4 with the columns `Kp`, `Ki`, `Kd`, `ITAE`,
  `evaluations`: one row per run of `run_ga(cost, BOUNDS_LO, BOUNDS_HI, seed=s)`, each with its own
  `cost = CostCounter(itae_cost)`, and `cost.evaluations` after the run;
- `GA`: the gains `(Kp, Ki, Kd)` of the row with the least ITAE.

In [ ]:
def blend_crossover(a, b, alpha, rng):
    ...


def mutate(child, lo, hi, p_mut, rng):
    ...


ga_table = ...
GA = ...

In [ ]:
# Tests for Task 4 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_crossover_stays_between_parents_without_alpha():
    rng = np.random.default_rng(0)
    a, b = np.array([1.0, 0.0, 10.0]), np.array([3.0, 0.1, 50.0])
    kids = np.array([blend_crossover(a, b, 0.0, rng) for _ in range(500)])
    assert kids.shape == (500, 3), "one child per call, the shape of a parent"
    above_low = (kids >= np.minimum(a, b) - 1e-12).all()
    below_high = (kids <= np.maximum(a, b) + 1e-12).all()
    assert above_low and below_high, "with alpha 0 inside the parents' box"


test_crossover_stays_between_parents_without_alpha()


def test_crossover_spreads_with_alpha():
    rng = np.random.default_rng(1)
    kids = [blend_crossover(np.zeros(5), np.ones(5), 0.25, rng) for _ in range(4000)]
    w = np.array(kids).ravel()             # with parents 0 and 1 the child is the weight
    in_range = w.min() >= -0.25 - 1e-12 and w.max() <= 1.25 + 1e-12
    assert in_range, "with alpha 0.25 the weights lie in [-0.25, 1.25]"
    assert w.min() < -0.2 and w.max() > 1.2, "and they fill that range"
    uniform = abs(w.mean() - 0.5) < 0.02 and abs(w.var() - 1.5 ** 2 / 12) < 0.01
    assert uniform, "uniform: mean 0.5, variance 1.5^2 / 12"


test_crossover_spreads_with_alpha()


def test_mutation_rate_and_size():
    lo, hi = np.zeros(3), np.array([1e6, 1e6, 1e6])
    c = np.full(3, 5e5)
    unchanged = mutate(c, lo, hi, 0.0, np.random.default_rng(2))
    assert np.array_equal(unchanged, c), "p_mut 0: no gene changes"
    children = [mutate(c, lo, hi, 1.0, np.random.default_rng(s)) for s in range(400)]
    shifts = (np.array(children) - c).ravel()
    assert (shifts != 0).all(), "p_mut 1: every gene changes"
    ok = abs(shifts.std() / 1e5 - 1) < 0.1
    assert ok, "the shift has a standard deviation of 10 % of the range"
    children = [mutate(c, lo, hi, 0.3, np.random.default_rng(s)) for s in range(400)]
    changed = np.array([child != c for child in children])
    assert abs(changed.mean() - 0.3) < 0.05, "each gene mutates with probability p_mut"


test_mutation_rate_and_size()


def test_mutation_clips_and_reproduces():
    lo, hi = np.array([0.1, 0.0, 0.0]), np.array([20.0, 0.2, 300.0])
    children = [mutate(hi.copy(), lo, hi, 1.0, np.random.default_rng(s))
                for s in range(200)]
    m = np.array(children)
    assert (m >= lo).all() and (m <= hi).all(), "the child is clipped to the box"
    c = np.array([5.0, 0.1, 100.0])
    first = mutate(c, lo, hi, 0.5, np.random.default_rng(3))
    second = mutate(c, lo, hi, 0.5, np.random.default_rng(3))
    assert np.array_equal(first, second), "the same seed gives the same child"
    assert np.array_equal(c, [5.0, 0.1, 100.0]), "mutate must not change its argument"


test_mutation_clips_and_reproduces()


def test_ga_finds_the_centre_of_a_bowl():
    centre = np.array([12.0, 0.05, 80.0])
    span = BOUNDS_HI - BOUNDS_LO
    bowl = lambda P: (((P - centre) / span) ** 2).sum(1)
    best, f, curve = run_ga(bowl, BOUNDS_LO, BOUNDS_HI, seed=0)
    found = np.all(np.abs(best - centre) < 0.03 * span)
    msg = f"run_ga with your operators finds the centre {centre}, got {np.round(best, 3)}"
    assert found, msg


test_ga_finds_the_centre_of_a_bowl()


def test_ga_results_reported():
    assert isinstance(ga_table, pd.DataFrame), "ga_table: a DataFrame"
    assert list(ga_table.index) == [0, 1, 2, 3, 4], "ga_table: index seeds 0-4"
    assert {"Kp", "Ki", "Kd", "ITAE", "evaluations"} <= set(ga_table.columns)
    full_budget = (ga_table["evaluations"] == POP * GENERATIONS).all()
    assert full_budget, "every run uses 1500 evaluations"
    assert len(GA) == 3, "GA: the gains (Kp, Ki, Kd)"


test_ga_results_reported()


print("all tests passed")

**Questions** (answer them in `practice_6_answers.md`, Task 4, with the numbers your code printed):

1. Print `ga_table`, and the overshoot and settling time of `GA` on the model. How much lower is its ITAE than that of SIMC and of ZN?
2. How far apart are the five results, in ITAE and in gains? Do different seeds find different tunings with the same ITAE, or the same tuning?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

*Common ground for Task 5: the particle swarm of In-class work 10, Part 6, except for its step,
which you write. `particles * iterations` evaluations. Run it as it is.*

In [19]:
def run_pso(cost, lo, hi, particles=POP, iterations=GENERATIONS, w=0.7, c1=1.5, c2=1.5,
            seed=0):
    # Returns the swarm's best tuning, its cost and the best cost after each iteration.
    rng = np.random.default_rng(seed)
    X = lo + (hi - lo) * rng.random((particles, len(lo)))
    V = np.zeros_like(X)
    f = cost(X)
    P, fP = X.copy(), f.copy()                 # every particle's own best and its cost
    g = P[np.argmin(fP)].copy()                # the swarm's best
    curve = [fP.min()]

    for _ in range(iterations - 1):
        X, V = pso_step(X, V, P, g, w, c1, c2, lo, hi, rng)
        f = cost(X)

        # update the memories
        better = f < fP
        P[better], fP[better] = X[better], f[better]
        g = P[np.argmin(fP)].copy()
        curve.append(fP.min())
    return g, float(fP.min()), np.array(curve)

---

## Task 5. Particle swarm, and the equal budget  
*0.25 points: 0.13 for the code (tests), 0.12 for the answers*

One PSO step for all particles at once. The rows of $\mathbf{X}$, $\mathbf{V}$ and $\mathbf{P}$ are
the particles' positions, velocities and own best positions, and $\mathbf{g}$ is the swarm's best:

$$\mathbf{V} \leftarrow w \mathbf{V} + c_1 \mathbf{R}_1 \odot (\mathbf{P} - \mathbf{X}) + c_2 \mathbf{R}_2 \odot (\mathbf{g} - \mathbf{X}), \qquad \mathbf{X} \leftarrow \text{clip}(\mathbf{X} + \mathbf{V}).$$

$\mathbf{g} - \mathbf{X}$ subtracts the row $\mathbf{g}$ from every row. $\mathbf{R}_1$ and
$\mathbf{R}_2$ are two independent uniform random matrices of the same shape as $\mathbf{X}$. The
inertia $w < 1$ damps the motion, and the two pulls make each particle swing around a point
between its own best and the swarm's best. The velocity is not clipped, only the position.

**Write**

- `pso_step(X, V, P, g, w, c1, c2, lo, hi, rng)`: the new `(X, V)`, every random number from `rng`,
  the arguments unchanged;
- `pso_table` and `PSO`: as `ga_table` and `GA` of Task 4, with `run_pso(cost, BOUNDS_LO, BOUNDS_HI,
  seed=s)`.

In [ ]:
def pso_step(X, V, P, g, w, c1, c2, lo, hi, rng):
    ...


pso_table = ...
PSO = ...

In [ ]:
# Tests for Task 5 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_no_pull_at_the_best():
    X = np.array([[1.0, 0.01, 10.0], [5.0, 0.1, 200.0]])
    V = np.array([[0.5, 0.0, -5.0], [0.0, 0.05, 50.0]])
    rng = np.random.default_rng(0)
    P, g = X.copy(), X[0].copy()          # particle 0: its own best and the swarm's
    X1, V1 = pso_step(X, V, P, g, 0.7, 1.5, 1.5, BOUNDS_LO, BOUNDS_HI, rng)
    ok = np.allclose(V1[0], 0.7 * V[0])
    assert ok, "a particle at its own best and at the swarm's best keeps only its inertia"
    moved = np.clip(X[0] + 0.7 * V[0], BOUNDS_LO, BOUNDS_HI)
    assert np.allclose(X1[0], moved), "and moves by it"


test_no_pull_at_the_best()


def test_pull_is_between_zero_and_c1_plus_c2():
    rng = np.random.default_rng(1)
    X, V = np.zeros((2000, 3)), np.ones((2000, 3))
    g = np.array([2.0, 0.1, 50.0])
    P = np.tile(g, (2000, 1))
    no_lo, no_hi = -1e9 * np.ones(3), 1e9 * np.ones(3)     # too wide to clip anything
    X1, V1 = pso_step(X, V, P, g, 0.5, 1.5, 1.5, no_lo, no_hi, rng)
    k = (V1 - 0.5 * V) / (g - X)                          # the pull per unit of g - X
    in_range = (k >= 0).all() and (k <= 3.0).all()
    assert in_range, "with P = g the pull is between 0 and c1 + c2 times g - X"
    assert abs(k.mean() - 1.5) < 0.05, "on average (c1 + c2) / 2"
    assert np.allclose(X1, X + V1), "the position moves by the new velocity"


test_pull_is_between_zero_and_c1_plus_c2()


def test_position_is_clipped():
    X = np.array([[19.0, 0.19, 290.0]])
    V = np.array([[100.0, 1.0, 1000.0]])
    rng = np.random.default_rng(2)
    P, g = X.copy(), X[0].copy()
    X1, V1 = pso_step(X, V, P, g, 0.7, 1.5, 1.5, BOUNDS_LO, BOUNDS_HI, rng)
    at_edge = np.allclose(X1, BOUNDS_HI)
    assert at_edge, "a particle that would leave the box stops at its edge"


test_position_is_clipped()


def test_pso_finds_the_centre_of_a_bowl():
    centre = np.array([12.0, 0.05, 80.0])
    span = BOUNDS_HI - BOUNDS_LO
    bowl = lambda P: (((P - centre) / span) ** 2).sum(1)
    best, f, curve = run_pso(bowl, BOUNDS_LO, BOUNDS_HI, seed=0)
    found = np.all(np.abs(best - centre) < 0.01 * span)
    msg = f"run_pso with your step finds the centre {centre}, got {np.round(best, 3)}"
    assert found, msg


test_pso_finds_the_centre_of_a_bowl()


def test_pso_results_reported():
    assert isinstance(pso_table, pd.DataFrame), "pso_table: a DataFrame"
    assert list(pso_table.index) == [0, 1, 2, 3, 4], "pso_table: index seeds 0-4"
    assert {"Kp", "Ki", "Kd", "ITAE", "evaluations"} <= set(pso_table.columns)
    full_budget = (pso_table["evaluations"] == POP * GENERATIONS).all()
    assert full_budget, "every run uses 1500 evaluations"
    assert len(PSO) == 3, "PSO: the gains (Kp, Ki, Kd)"


test_pso_results_reported()


print("all tests passed")

**Questions** (answer them in `practice_6_answers.md`, Task 5, with the numbers your code printed):

1. Print `pso_table`, and the overshoot and settling time of `PSO` on the model.
2. One figure: the median and the range over the five seeds of the best-so-far ITAE against the number of evaluations, GA and PSO together (both use 30 evaluations per generation or iteration; a log scale on ITAE helps).
3. Which method is better *on this budget*: in the median, in the worst seed, in speed? Do GA and PSO agree on the gains?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 6. On the real exchanger  
*0.15 points: 0.06 for the code (tests), 0.09 for the answers*

Every tuning so far was made and scored on the model. Now check the four on the exchanger itself:
`closed_loop(Kp, Ki, Kd)` from `exchanger.py` gives the response in the same deviation units as
`simulate_loop`, and takes arrays of gains too.

**Write** two `pd.DataFrame`s indexed by `"ZN"`, `"SIMC"`, `"GA"`, `"PSO"` with the columns
`"overshoot"`, `"settling"`, `"ITAE"` of `step_metrics`:

- `model_table`: the responses of `simulate_loop` with each tuning;
- `exchanger_table`: the responses of `closed_loop` with each tuning.

In [ ]:
model_table = ...
exchanger_table = ...

In [ ]:
# Tests for Task 6 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_tables_reported():
    tables = {"model_table": model_table, "exchanger_table": exchanger_table}
    for name, tab in tables.items():
        assert isinstance(tab, pd.DataFrame), f"{name}: a DataFrame"
        in_order = list(tab.index) == ["ZN", "SIMC", "GA", "PSO"]
        assert in_order, f"{name}: index ZN, SIMC, GA, PSO"
        has_columns = {"overshoot", "settling", "ITAE"} <= set(tab.columns)
        assert has_columns, f"{name}: columns overshoot, settling, ITAE"


test_tables_reported()


print("all tests passed")

**Questions** (answer them in `practice_6_answers.md`, Task 6, with the numbers your code printed):

1. Print one table for the four tunings: the gains, and overshoot, settling time and ITAE on the model and on the exchanger. Plot the four responses of the exchanger.
2. Did the ranking survive the move from the model to the plant? Which tunings got better and which got worse, and what in Task 1 could have warned you?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 7. Robustness to the dead time  
*0.15 points: 0.07 for the code (tests), 0.08 for the answers*

In winter the steam line is longer in use and the dead time of the exchanger grows. A tuning that
is best at one dead time can oscillate at another. The **robust cost** of a tuning is its worst ITAE
over three models with the dead time $L$, $1.5L$ and $2L$:

$$J_\text{robust}(K_p, K_i, K_d) = \max_{f \in \{1,\ 1.5,\ 2\}} \text{ITAE}(K_p, K_i, K_d;\ f L).$$

Minimising the worst case gives up some performance at the nominal dead time to stay calm at the
longer ones.

**Write**

- `robust_cost(P)`: for an array of tunings `(k, 3)`, the array `(k,)` of their robust costs on the
  model (`itae_cost(P, L=...)` with `L_M`, `1.5 * L_M`, `2 * L_M`);
- `robust_table`: a `pd.DataFrame` indexed by seed 0–4 with the columns `Kp`, `Ki`, `Kd`,
  `"worst ITAE"`, one row per `run_pso(robust_cost, BOUNDS_LO, BOUNDS_HI, seed=s)`;
- `ROBUST`: the gains `(Kp, Ki, Kd)` of the row with the least worst ITAE.

Five robust runs take about two minutes.

In [ ]:
def robust_cost(P):
    ...


robust_table = ...
ROBUST = ...

In [ ]:
# Tests for Task 7 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_robust_cost_is_a_worst_case():
    P = np.array([[1.0, 0.01, 0.0], [2.0, 0.02, 10.0]])
    got = np.asarray(robust_cost(P))
    assert got.shape == (2,), f"one robust cost per tuning, got shape {got.shape}"
    nominal = itae_cost(P)
    assert np.all(got >= nominal - 1e-9), "the worst ITAE is at least the nominal one"
    at_double = itae_cost(P, L=2 * L_M)
    assert np.all(got >= at_double - 1e-9), "and at least the one at twice the dead time"


test_robust_cost_is_a_worst_case()


def test_robust_results_reported():
    assert isinstance(robust_table, pd.DataFrame), "robust_table: a DataFrame"
    assert list(robust_table.index) == [0, 1, 2, 3, 4], "robust_table: index seeds 0-4"
    assert {"Kp", "Ki", "Kd", "worst ITAE"} <= set(robust_table.columns)
    assert len(ROBUST) == 3, "ROBUST: the gains (Kp, Ki, Kd)"


test_robust_results_reported()


print("all tests passed")

**Questions** (answer them in `practice_6_answers.md`, Task 7, with the numbers your code printed):

1. On the model, run the four tunings with the dead time $L$ times 1.25, 1.5 and 2. Print a table of overshoot and settling time. Which tunings stop settling?
2. Print `robust_table`. Did every seed find the same tuning? What does that say about the robust cost?
3. Compare `ROBUST` with `PSO` and with `SIMC` on the model at $L$, $1.5L$, $2L$, and on the exchanger with `extra_dead_time` 0 and 35 s. What does robustness cost on the nominal plant?
4. Your recommendation and conclusions, in eight to twelve sentences: what GA and PSO gained over Ziegler–Nichols and over SIMC, in numbers, on the model and on the exchanger; what the choice between GA and PSO changed, compared with the choice of the cost function; which tuning goes into the controller, and what you would ask the plant engineers to measure first.

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

*Common ground for the optional Task 8: the network, its four plant parameters and the training
loop. It needs PyTorch and runs on a GPU if PyTorch finds one; the first line it prints says
which. Run it as it is.*

In [29]:
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("the PINN trains on", DEVICE)
W_STEP = 2.0       # s: the valve step is smoothed over 2 s, so that L gets a gradient


def valve(t, L):
    # the valve above 40 %, delayed by L: a smoothed step of DU at T_STEP
    return DU * torch.sigmoid((t - T_STEP - L) / W_STEP)


class PINN(torch.nn.Module):
    # y(t), the outlet above its value before the step, as a small network of t,
    # plus the four parameters of equation (1)
    def __init__(self):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(1, 32),
            torch.nn.Tanh(),
            torch.nn.Linear(32, 32),
            torch.nn.Tanh(),
            torch.nn.Linear(32, 32),
            torch.nn.Tanh(),
            torch.nn.Linear(32, 1),
        )
        # log K, log T1, log T2, log L: the logs keep them positive. The start is far from
        # any answer on purpose.
        start = torch.tensor([1.0, 60.0, 10.0, 30.0])
        self.log_p = torch.nn.Parameter(torch.log(start))

    def forward(self, t):                     # t in seconds, shape (n, 1)
        # scaled so that the network sees numbers of order 1
        return 8.0 * self.net(t / 900.0)

    def params(self):
        return torch.exp(self.log_p)          # K, T1, T2, L


def train_pinn(t_data, y_data, seed=0, steps=5000, lam=1.0):
    # Trains on DEVICE; t_data, y_data (n, 1) may live anywhere. Returns the model,
    # still on DEVICE, and the history of the losses and the parameters every 100 steps.
    torch.manual_seed(seed)
    model = PINN().to(DEVICE)
    t_data, y_data = t_data.to(DEVICE), y_data.to(DEVICE)
    t_col = torch.linspace(0.0, 900.0, 600, device=DEVICE)[:, None]   # collocation, s
    opt = torch.optim.Adam([{"params": model.net.parameters(), "lr": 3e-3},
                            {"params": [model.log_p], "lr": 2e-2}])

    history = []
    for k in range(steps + 1):
        opt.zero_grad()
        loss, data, phys = pinn_loss(model, t_data, y_data, t_col, lam)
        loss.backward()
        opt.step()
        if k % 100 == 0:
            history.append([k, data.item(), phys.item(), *model.params().tolist()])
    columns = ["step", "data", "phys", "K", "T1", "T2", "L"]
    return model, pd.DataFrame(history, columns=columns)

the PINN trains on cuda


---

## Task 8 (optional). A physics-informed network finds the second lag  
*Optional, no points: extra work for those who want it*

Task 1 showed that a first order plus dead time model misses the slow, S-shaped start of the step
test, and Task 6 paid for it. Two lags in series describe that start:

$$T_1 T_2\,\ddot y + (T_1 + T_2)\,\dot y + y = K\,\Delta u(t - L) \qquad (1)$$

where $y$ is the outlet above its value before the step and $\Delta u$ the valve above 40 %. Right
after the dead time the first lag has barely started to move, so the second one sees almost no
input and the output rises with zero slope: that is the S.

A **physics-informed neural network** fits a small network $y_{\boldsymbol{\theta}}(t)$ to the data
and at the same time makes it satisfy (1) at 600 collocation points spread over the 15 minutes.
$K$, $T_1$, $T_2$ and $L$ are trained together with the weights $\boldsymbol{\theta}$ of the
network, so the training returns a model of the plant as well as a curve through the points.

The residual of (1) at time $t$:

$$r(t) = T_1 T_2\, y''(t) + (T_1 + T_2)\, y'(t) + y(t) - K\,\Delta u(t - L),$$

with $\Delta u(t - L)$ = `valve(t, L)`. Where the network obeys the physics, $r = 0$; the size of
$r$ says how far the curve is from a solution of (1) with the current parameters. The derivatives
come from autograd: `torch.autograd.grad(y.sum(), t, create_graph=True)[0]` is $dy/dt$ at every
point of `t`. The sum works because each $y_k$ depends on its own $t_k$ only. `create_graph=True`
keeps the graph, so that you can differentiate a second time and the loss can be backpropagated
through the derivatives.

The loss over the data points $(t_i, y_i)$ and the collocation points $t_c$:

$$\mathcal L(\boldsymbol{\theta}, K, T_1, T_2, L) = \frac{1}{N}\sum_i \big(y_{\boldsymbol{\theta}}(t_i) - y_i\big)^2 + \lambda\,\frac{1}{M}\sum_c r(t_c)^2.$$

The first term pulls the curve through the data, the second pulls the curve and the parameters
towards each other: the only way to make both small is a curve that fits the data *and* solves (1),
and then the parameters are the plant's. $\lambda$ sets how hard the physics pulls.

**Write**

- `residual(y_fn, p, t)`: $r$ at the times `t` (a tensor `(n, 1)`, seconds) for the function
  `y_fn` (a tensor `(n, 1)` to `(n, 1)`) and `p = (K, T1, T2, L)`; a tensor `(n, 1)` that can be
  backpropagated;
- `pinn_loss(model, t_data, y_data, t_col, lam=1.0)`: `(loss, data, phys)`, three scalar tensors:
  the total and its two terms; `train_pinn` above calls it;
- `PINN_P`: `(K, T1, T2, L)` of `train_pinn` with seed 0 and $\lambda$ = 1 on the step test, with
  the larger lag reported as `T1` (equation (1) does not change if they swap). For the data take
  the outlet minus its mean before the step.

In [ ]:
def residual(y_fn, p, t):
    ...


def pinn_loss(model, t_data, y_data, t_col, lam=1.0):
    ...


PINN_P = ...

In [ ]:
# Tests for Task 8 (optional) (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_residual_of_an_exact_solution():
    def exact(t, K=2.0, T1=50.0, T2=10.0, L=12.0):
        # the step response of (1) with these parameters
        s = torch.clamp(t - T_STEP - L, min=0.0)
        lags = (T1 * torch.exp(-s / T1) - T2 * torch.exp(-s / T2)) / (T1 - T2)
        return K * DU * (1 - lags)


    # times away from the step, where the residual of the exact response must vanish
    t_chk = torch.cat([torch.arange(0.0, 50.0), torch.arange(100.0, 600.0)])[:, None]
    right = torch.tensor([2.0, 50.0, 10.0, 12.0])
    wrong = torch.tensor([2.0, 50.0, 20.0, 12.0])
    r_ok = residual(exact, right, t_chk).abs().max().item()
    r_bad = residual(exact, wrong, t_chk).abs().max().item()
    msg = f"with the right parameters the residual vanishes there, got {r_ok:.1e}"
    assert r_ok < 1e-3, msg
    assert r_bad > 1, f"with T2 = 20 s instead of 10 it does not, got {r_bad:.2f}"


test_residual_of_an_exact_solution()


def test_loss_of_a_tiny_model():
    class Parabola(torch.nn.Module):        # y = 0.001 t^2, with fixed parameters
        def forward(self, t):
            return 0.001 * t ** 2

        def params(self):
            return torch.tensor([1.0, 2.0, 3.0, 0.0])


    t_data = torch.tensor([[0.0], [10.0]])
    y_data = torch.tensor([[1.0], [1.0]])
    t_col = torch.tensor([[0.0], [900.0]])
    loss, data, phys = pinn_loss(Parabola(), t_data, y_data, t_col, lam=2.0)
    misfit = data.item()
    msg = f"mean squared misfit: ((0 - 1)^2 + (0.1 - 1)^2) / 2 = 0.905, got {misfit}"
    assert np.isclose(misfit, 0.905), msg
    r = residual(Parabola(), Parabola().params(), t_col).detach()
    ok = np.isclose(phys.item(), (r ** 2).mean().item(), rtol=1e-5)
    assert ok, "phys: the mean squared residual at t_col"
    ok = np.isclose(loss.item(), data.item() + 2.0 * phys.item(), rtol=1e-5)
    assert ok, "loss = data + lam * phys"


test_loss_of_a_tiny_model()


def test_pinn_parameters_reported():
    positive = len(PINN_P) == 4 and all(float(v) > 0 for v in PINN_P)
    assert positive, "PINN_P: four positive numbers K, T1, T2, L"


test_pinn_parameters_reported()


print("all tests passed")

**Questions** (answer them in `practice_6_answers.md`, Task 8 (optional), with the numbers your code printed):

1. Train the PINN on the step test with seeds 0, 1, 2 and $\lambda$ = 1. For each seed print $K$, $T_1$, $T_2$, $L$ and the RMS misfit to the data.
2. Check it by plain least squares: `curve_fit` of the exact step response of (1), $y = K\Delta u\,[1 - (T_1 e^{-s/T_1} - T_2 e^{-s/T_2})/(T_1 - T_2)]$ with $s = \max(0,\ t - t_s - L)$. Do the two agree?
3. Train seed 0 once more with $\lambda$ = 10. What happens to the parameters, and why?
4. One figure: the data, the PINN curve and the least-squares FOPDT of Task 1, with an inset on the first two minutes; next to it $T_1$, $T_2$, $L$ against the training step for $\lambda$ = 1 and 10. What do the two panels show?
5. Retune on the new model. Write `simulate_loop2(Kp, Ki, Kd, K, T1, T2, L)`, the PID loop on (1) ($a_i = e^{-\Delta t/T_i}$, $x_1 \leftarrow a_1 x_1 + (1-a_1) K u$, $x_2 \leftarrow a_2 x_2 + (1-a_2) x_1$, the measurement is $x_2$; pass that step to `_pid_loop` with two states), and run `run_pso` (seed 0, the same 1500 evaluations) on its ITAE. Call the result `PSO2` and check it on the exchanger with `extra_dead_time` 0 and 35 s, next to PSO and ROBUST.
6. Five to eight sentences. Where did the gain over Task 6 come from, the optimiser or the model? What did the PINN give here that `curve_fit` did not, and where would it give more? Did the better model fix the dead-time problem of Task 7?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers